### Mini Project: Predicting Employee Attrition Using Classification Models on the IBM HR Analytics Dataset

Note: This is a graded assignment and contributes to your overall course evaluation. You’ll apply key data science techniques to a real-world problem dataset to demonstrate your analytical and modelling abilities. 


Learning outcomes addressed:

Perform exploratory data analysis using Python libraries
Apply classification techniques and evaluate model performance
Handle data preprocessing tasks including class imbalance and feature engineering
Overview: Build a classification model to identify if an employee is likely to leave the organisation using HR data.

What this is about:
You’ll work with the IBM HR Analytics dataset containing attributes such as satisfaction level, job role, work-life balance, salary, and overtime status. The goal is to:

1. Explore patterns in attrition
1. Handle class imbalance
1. Train multiple classification models
1. Evaluate using suitable metrics
1. Extract and report key attrition driver

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('data.csv')

## Step 1: Exploratory Data Analysis (EDA)

The objective of EDA is to understand the structure, quality, and characteristics of the dataset and identify factors that may influence employee attrition.

#### Dataset Size

Understand the overall dimensions of the dataset by examining the number of rows and columns.

In [ ]:
df.shape

#### Dataset Preview

Inspect a sample of records to gain familiarity with the available features and their values.

In [ ]:
df.sample(5)

#### Data Types

Review the data types of each feature to determine the appropriate preprocessing and analysis techniques.

In [ ]:
df.info()

#### Missing Value Analysis

Check for null or missing values that may require treatment before model development.

In [ ]:
df.isnull().sum()

#### Statistical Summary

Examine the central tendency, spread, and distribution of numerical features.

In [ ]:
df.describe()

#### Duplicate Record Check

Identify duplicate observations that could introduce bias into the analysis.

In [ ]:
df.duplicated().sum()

#### Correlation Analysis

Analyze relationships between numerical features and identify potentially redundant variables.

In [ ]:
df.select_dtypes(include=['number']).corr()

#### Univariate Analysis

Analyze each feature independently to understand its distribution and characteristics.

##### Analysis of Categorical Features

Explore the distribution of categorical variables across the employee population.

In [ ]:

df.select_dtypes(include=['object']).nunique().sort_values(ascending=False)


> **Note**
>
> - **`Over18`** has been excluded from the analysis as it contains only a single unique value and does not provide any meaningful information.
> - **`Attrition`** has been excluded from the univariate categorical analysis since it is the target va

In [ ]:

categorical_columns = [
    col for col in df.select_dtypes(include=['object']).columns
    if col not in ['Attrition', 'Over18']
]

n_cols = 3
n_rows = math.ceil(len(categorical_columns) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(24, 6*n_rows)
)

# Overall title
fig.suptitle(
    'Distribution of Categorical Features',
    fontsize=20,
    fontweight='bold',
    y=1.02
)

axes = axes.flatten()

for i, col in enumerate(categorical_columns):

    sns.countplot(
        data=df,
        x=col,
        hue=col,
        palette='viridis',
        legend=False,
        ax=axes[i]
    )

    axes[i].tick_params(axis='x', rotation=45)

# Remove empty plots
for j in range(len(categorical_columns), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
df['Attrition'].value_counts().plot(
    kind='bar',
    color=['green', 'red']
)

plt.title('Attrition Distribution')
plt.xlabel('Attrition Status')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

df['Attrition'].value_counts().plot(
    kind='pie',
    colors=['green', 'red'],
    autopct='%.2f%%',
    startangle=90
)

plt.title('Employee Attrition Distribution', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.show()

In [ ]:
numerical_columns = df.select_dtypes(include=['number']).columns.tolist()

n_cols = 3
n_rows = (len(numerical_columns) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 5 * n_rows)
)

axes = axes.flatten()

for i, col in enumerate(numerical_columns):
    sns.histplot(
        df[col],
        bins=20,
        kde=True,
        ax=axes[i]
    )
    axes[i].set_title(col)

# Remove unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    'Distribution of Numerical Features',
    fontsize=22,
    fontweight='bold'
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

#### Bivariate Analysis

Study the relationship between employee attrition and other features to identify potential attrition drivers.

##### Categorical Features vs Attrition

Evaluate how attrition varies across departments, job roles, overtime, marital status, and other categorical variables.

##### Numerical Features vs Attrition

Compare numerical feature distributions between employees who stayed and employees who left the organization.

In [ ]:
continuous_numerical_features = [
    "Age",
    "DailyRate",
    "DistanceFromHome",
    "HourlyRate",
    "MonthlyIncome",
    "MonthlyRate",
    "PercentSalaryHike",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]

n_cols = 3
n_rows = math.ceil(len(continuous_numerical_features) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 5 * n_rows)
)

# Overall title
fig.suptitle(
    'Continuous Numerical Features vs Attrition',
    fontsize=20,
    fontweight='bold'
)

axes = axes.flatten()

for i, col in enumerate(continuous_numerical_features):

    sns.boxplot(
        data=df,
        x='Attrition',
        y=col,
        hue='Attrition',
        palette='viridis',
        legend=False,
        ax=axes[i]
    )

    axes[i].set_title(col)
    axes[i].set_xlabel('Attrition')

# Remove unused subplots
for j in range(len(continuous_numerical_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
numerical_categorical_features = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TrainingTimesLastYear",
    "WorkLifeBalance"
]

n_cols = 3
n_rows = math.ceil(len(numerical_categorical_features) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 5 * n_rows)
)

axes = axes.flatten()

for i, col in enumerate(numerical_categorical_features):

    sns.countplot(
        data=df,
        x=col,
        hue='Attrition',
        palette='viridis',
        ax=axes[i]
    )

    axes[i].set_title(f'{col} vs Attrition')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

# Remove unused subplots
for j in range(len(numerical_categorical_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

#### Observation

The correlation analysis indicates a few moderately correlated feature groups, particularly among tenure-related variables (YearsAtCompany, YearsInCurrentRole, YearsWithCurrManager) and compensation-related variables (JobLevel and MonthlyIncome).

No features were removed based solely on correlation, as the observed relationships are expected from a business perspective and the feature set remains manageable for model training.

In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    df.select_dtypes(include='number').corr(),
    cmap='coolwarm',
    center=0
)

plt.title(
    'Correlation Heatmap of Numerical Features',
    fontsize=16,
    fontweight='bold'
)

plt.show()

## Step 2: Data Preprocessing

Prepare the dataset for machine learning by applying feature encoding and scaling techniques.

### Feature Encoding

Categorical features were encoded based on their characteristics:

- **Ordinal:** BusinessTravel
- **One-Hot:** Department, EducationField, JobRole, MaritalStatus
- **Binary:** Gender, OverTime
- **Target Encoding:** Attrition (No = 0, Yes = 1)

The selected encoding techniques preserve the semantic meaning of each feature while preparing the dataset for model training.

In [ ]:
# remove not required columns - EmployeeCount, EmployeeNumber, over18, StandardHours
df.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'], inplace=True)

In [ ]:
# train/test/split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['Attrition']),
    df['Attrition'],
    test_size=0.2,
    random_state=42)

In [ ]:
# Encode Target Variable
y_train = y_train.map({
    'No': 0,
    'Yes': 1
})

y_test = y_test.map({
    'No': 0,
    'Yes': 1
})

In [ ]:
# feature groups

# ordinal encoding for the following features

ordinal_features = [
    'BusinessTravel'
]


ordinal_encoder = OrdinalEncoder(
    categories=[
        [
            'Non-Travel',
            'Travel_Rarely',
            'Travel_Frequently'
        ]
    ]
)

# binary encoding for the following features

binary_features = [
    'Gender',
    'OverTime'
]
binary_encoder = OrdinalEncoder()

# one hot encoding for the following features
one_hot_features = [
    'Department',
    'EducationField',
    'JobRole',
    'MaritalStatus'
]

one_hot_encoder = OneHotEncoder(
    drop='first',
    handle_unknown='ignore',
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'ordinal',
            ordinal_encoder,
            ordinal_features
        ),
        (
            'binary',
            binary_encoder,
            binary_features
        ),
        (
            'onehot',
            one_hot_encoder,
            one_hot_features
        )
    ],
    remainder='passthrough'
)

In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

In [ ]:
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=feature_names,
    index=X_train.index
)


X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=feature_names,
    index=X_test.index
)

### Feature Scaling

The features were standardized using **StandardScaler**, transforming them to have a mean of 0 and a standard deviation of 1. This helps algorithms converge efficiently and prevents features with larger magnitudes from dominating the learning process.

In [ ]:
# Initialize scaler
scaler = StandardScaler()

# Fit on training data and transform
X_train_scaled = scaler.fit_transform(X_train_encoded)

# Transform test data
X_test_scaled = scaler.transform(X_test_encoded)

In [ ]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train_encoded.columns,
    index=X_train_encoded.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test_encoded.columns,
    index=X_test_encoded.index
)

X_train_scaled.head()

## Step 3: Model Building

The preprocessed dataset is now ready for training and evaluating classification models. Multiple algorithms will be compared using appropriate performance metrics to identify the best model for predicting employee attrition.